In [5]:
import numpy as np

# Activation functions
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def tanh(Z):
    return np.tanh(Z)

def sigmoid_derivative(A):
    return A * (1 - A)

def tanh_derivative(A):
    return 1 - A**2


# Initialize parameters
def initialize():
    np.random.seed(42)
    W1 = np.random.randn(2, 2) * 0.1
    b1 = np.zeros((2, 1))
    W2 = np.random.randn(1, 2) * 0.1
    b2 = np.zeros((1, 1))
    return W1, b1, W2, b2


# Forward propagation
def forward(X, W1, b1, W2, b2):
    Z1 = np.dot(W1, X) + b1
    A1 = tanh(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2


# Loss (Binary Cross Entropy)
def compute_loss(A2, Y):
    m = Y.shape[1]
    loss = -np.sum(Y*np.log(A2) + (1-Y)*np.log(1-A2)) / m
    return loss


# Backpropagation
def backward(X, Y, Z1, A1, Z2, A2, W2):
    m = X.shape[1]

    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * tanh_derivative(A1)
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2


# Training
def train(X, Y, epochs=10000, lr=0.1):
    W1, b1, W2, b2 = initialize()

    for i in range(epochs):
        Z1, A1, Z2, A2 = forward(X, W1, b1, W2, b2)

        loss = compute_loss(A2, Y)

        dW1, db1, dW2, db2 = backward(X, Y, Z1, A1, Z2, A2, W2)

        # Update
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

        if i % 1000 == 0:
            print(f"Epoch {i}, Loss: {loss:.4f}")

    return W1, b1, W2, b2


# Prediction
def predict(X, W1, b1, W2, b2):
    _, _, _, A2 = forward(X, W1, b1, W2, b2)
    return (A2 > 0.5).astype(int)


# XOR dataset
X = np.array([
    [0, 0, 1, 1],
    [0, 1, 0, 1]
])

Y = np.array([[0, 1, 1, 0]])

# Train
W1, b1, W2, b2 = train(X, Y)

# Predict
preds = predict(X, W1, b1, W2, b2)
print("Predictions:", preds)

Epoch 0, Loss: 0.6932
Epoch 1000, Loss: 0.6931
Epoch 2000, Loss: 0.6931
Epoch 3000, Loss: 0.6931
Epoch 4000, Loss: 0.6930
Epoch 5000, Loss: 0.5849
Epoch 6000, Loss: 0.0661
Epoch 7000, Loss: 0.0225
Epoch 8000, Loss: 0.0133
Epoch 9000, Loss: 0.0094
Predictions: [[0 1 1 0]]


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)   # more neurons
        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))  # smoother than ReLU
        x = torch.sigmoid(self.fc2(x))
        return x


def train_xor():
    X = torch.tensor([
        [0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]
    ])
    
    y = torch.tensor([[0.], [1.], [1.], [0.]])

    model = XORNet()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()

    for epoch in range(5000):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

    return model


model = train_xor()

with torch.no_grad():
    preds = model(X)
    print((preds > 0.5).float())

tensor([[0.],
        [1.],
        [1.],
        [0.]])
